# Step 3: LC-Protonets

## Goals
1. Define a ResNet/ViT-based feature extractor.
2. Sample few-shot episodes.
3. Build dynamic prototypes for observed label combinations.
4. Train and run inference with LC-Protonets.
5. Evaluate the model on the validation set.

## Core Ideas
- Build prototypes for label combinations observed in each episode.
- Support multi-label data, where samples may contain multiple labels.
- Classify query samples by their distances to prototypes in feature space.

## Episode Setup
- Training: 52-way, 1- or 5-shot, 4 queries per class.
- Validation: 12-way, 1- or 5-shot, 4 queries per class.
- Test: 16-way, 1- or 5-shot, 4 queries per class.


## 3.1 Import Required Libraries


In [ ]:
import os
import json
import random
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from collections import defaultdict
from itertools import combinations

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

# Independent full-pipeline seed supplied by run_multiseed_experiment.py
EXPERIMENT_SEED = int(os.environ.get('EXPERIMENT_SEED', '42'))

def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    strict_determinism = os.environ.get('STRICT_DETERMINISM', '0') == '1'
    torch.backends.cudnn.deterministic = strict_determinism
    torch.backends.cudnn.benchmark = not strict_determinism
    if strict_determinism:
        torch.use_deterministic_algorithms(True, warn_only=True)

seed_everything(EXPERIMENT_SEED)
print(f'Experiment seed: {EXPERIMENT_SEED}')
print(f'Strict CUDA determinism: {os.environ.get("STRICT_DETERMINISM", "0")}')

PROGRESS_FILE = os.environ.get('RUN_PROGRESS_FILE')

def update_run_progress(phase, current, total, **details):
    if not PROGRESS_FILE:
        return
    payload = {
        'seed': EXPERIMENT_SEED,
        'phase': phase,
        'current': int(current),
        'total': int(total),
        'details': details
    }
    os.makedirs(os.path.dirname(PROGRESS_FILE), exist_ok=True)
    temporary_file = PROGRESS_FILE + '.tmp'
    with open(temporary_file, 'w', encoding='utf-8') as handle:
        json.dump(payload, handle)
    os.replace(temporary_file, PROGRESS_FILE)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

## 3.2 Configure Paths and Parameters


In [ ]:
DATASET_ROOT = os.environ.get("COCO_ROOT", "coco2014")
TRAIN2014_DIR = os.path.join(DATASET_ROOT, "train2014")
VAL2014_DIR = os.path.join(DATASET_ROOT, "val2014")
ANNOTATIONS_DIR = os.path.join(DATASET_ROOT, "annotations")

DATA_DIR = os.environ.get("DEPF_DATA_DIR", "coco2014_data")
DATA_SPLIT_FILE = os.path.join(DATA_DIR, "coco_mutually_exclusive_split.json")
CATEGORY_MAPPING_FILE = os.path.join(DATA_DIR, "category_mapping.json")

# Seed-isolated output path
MULTISEED_ROOT = os.environ.get(
    'MULTISEED_ROOT',
    'coco2014_multiseed_k5'
)
RUN_DIR = os.path.join(MULTISEED_ROOT, f'seed_{EXPERIMENT_SEED}')
OUTPUT_DIR = os.path.join(RUN_DIR, 'lc_protonets')
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f"Dataset root: {DATASET_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

## 3.3 Define Class Sets


In [ ]:
BASE_LABELS_COCO = [
    'hair drier', 'clock', 'wine glass', 'book', 'cake', 'tie', 'motorcycle', 'sheep',
    'bottle', 'giraffe', 'cell phone', 'suitcase', 'remote', 'bench', 'mouse', 'carrot',
    'banana', 'train', 'sports ball', 'toothbrush', 'fire hydrant', 'airplane', 'tv',
    'bus', 'refrigerator', 'couch', 'knife', 'toilet', 'elephant', 'truck', 'parking meter',
    'car', 'potted plant', 'kite', 'skateboard', 'orange', 'horse', 'cat', 'tennis racket',
    'bowl', 'scissors', 'baseball glove', 'apple', 'traffic light', 'handbag', 'donut',
    'dog', 'hot dog', 'oven', 'umbrella', 'sink', 'pizza'
]

VAL_LABELS_COCO = [
    'cow', 'dining table', 'zebra', 'sandwich', 'bear', 'toaster', 'person', 'laptop',
    'bed', 'teddy bear', 'baseball bat', 'skis'
]

NOVEL_LABELS_COCO = [
    'bicycle', 'boat', 'stop sign', 'bird', 'backpack', 'frisbee', 'snowboard', 'surfboard',
    'cup', 'fork', 'spoon', 'broccoli', 'chair', 'keyboard', 'microwave', 'vase'
]

print(f"Base classes: {len(BASE_LABELS_COCO)}")
print(f"Validation classes: {len(VAL_LABELS_COCO)}")
print(f"Novel classes: {len(NOVEL_LABELS_COCO)}")

## 3.4 Load the Dataset Split


In [ ]:
with open(DATA_SPLIT_FILE, 'r') as f:
    data_split = json.load(f)

with open(CATEGORY_MAPPING_FILE, 'r') as f:
    category_mapping = json.load(f)

base_samples = data_split['base_samples']
val_samples = data_split['val_samples']
novel_samples = data_split['novel_samples']

cat_name_to_id = category_mapping['cat_name_to_id']
cat_id_to_name = category_mapping['cat_id_to_name']

print(f"\nData split loaded:")
print(f"  Base samples: {len(base_samples)}")
print(f"  Val samples: {len(val_samples)}")
print(f"  Novel samples: {len(novel_samples)}")

## 3.5 Define the Multi-Label Dataset


In [ ]:
class MultiLabelCOCODataset(Dataset):
    """Load COCO images and multi-label targets for episodic learning."""
    
    def __init__(
        self,
        samples: List[Dict],
        train_image_dir: str,
        val_image_dir: str,
        transform=None,
        cat_id_to_name: Dict[int, str] = None
    ):
        self.samples = samples
        self.train_image_dir = train_image_dir
        self.val_image_dir = val_image_dir
        self.transform = transform
        self.cat_id_to_name = cat_id_to_name or {}
    
    def __len__(self):
        return len(self.samples)
    
    def _get_image_path(self, sample: Dict) -> str:
        """Find an image in the configured COCO image directories."""
        filename = sample.get('filename', f"{sample['image_id']:012d}.jpg")
        
        val_path = os.path.join(self.val_image_dir, filename)
        if os.path.exists(val_path):
            return val_path
        
        train_path = os.path.join(self.train_image_dir, filename)
        if os.path.exists(train_path):
            return train_path
        
        raise FileNotFoundError(f"COCO2014 image not found in train2014 or val2014: {filename}")
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        image_path = self._get_image_path(sample)
        image = Image.open(image_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        labels = torch.tensor(sample['category_ids'], dtype=torch.long)
        
        return {
            'image': image,
            'labels': labels,
            'image_id': sample['image_id'],
            'category_ids': sample['category_ids']
        }

print("Testing MultiLabelCOCODataset...")
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

test_dataset = MultiLabelCOCODataset(
    samples=val_samples[:10],
    train_image_dir=TRAIN2014_DIR,
    val_image_dir=VAL2014_DIR,
    transform=test_transform,
    cat_id_to_name=cat_id_to_name
)

test_sample = test_dataset[0]
print(f"Image shape: {test_sample['image'].shape}")
print(f"Labels: {test_sample['labels']}")
print(f"\nMultiLabelCOCODataset test passed!")

## 3.6 Define the Feature Extractor


In [ ]:
import torchvision.models as models

class FeatureExtractor(nn.Module):
    """Extract image embeddings with a ResNet-50 backbone and projection layer."""
    
    def __init__(self, pretrained=True, feature_dim=2048, finetune=True):
        super(FeatureExtractor, self).__init__()
        
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        
        if feature_dim != 2048:
            self.projection = nn.Sequential(
                nn.Flatten(),
                nn.Linear(2048, feature_dim),
                nn.ReLU(inplace=True)
            )
            self.use_projection = True
        else:
            self.projection = nn.Flatten()
            self.use_projection = False
        
        self.finetune = finetune
        
        if not finetune:
            for param in self.features.parameters():
                param.requires_grad = False
        
        print(f"FeatureExtractor initialized:")
        print(f"  Pretrained: {pretrained}")
        print(f"  Feature dimension: {feature_dim}")
        print(f"  Finetune ResNet: {finetune}")
        print(f"  Use projection layer: {self.use_projection}")
    
    def forward(self, x):
        """Run the input batch through the feature extractor."""
        x = self.features(x)  # (B, 2048, 1, 1)
        
        features = self.projection(x)  # (B, feature_dim)
        
        return features

print("\nTesting FeatureExtractor...")
feature_extractor = FeatureExtractor(pretrained=False, feature_dim=2048, finetune=True).to(DEVICE)

test_input = torch.randn(4, 3, 224, 224).to(DEVICE)
test_output = feature_extractor(test_input)

print(f"Input shape: {test_input.shape}")
print(f"Output shape: {test_output.shape}")
print(f"\nFeatureExtractor test passed!")

## 3.7 Define the Episode Sampler


In [ ]:
class EpisodeSampler:
    """Sample support and query examples for few-shot episodes."""
    
    def __init__(
        self,
        dataset: MultiLabelCOCODataset,
        n_way: int,
        k_shot: int,
        n_query: int = 4,
    ):
        self.dataset = dataset
        self.n_way = n_way
        self.k_shot = k_shot
        self.n_query = n_query
        
        self.cat_to_indices = defaultdict(list)
        for idx, sample in enumerate(dataset.samples):
            for cat_id in sample['category_ids']:
                self.cat_to_indices[cat_id].append(idx)
        
        self.available_cats = list(self.cat_to_indices.keys())
        
        self.image_to_cats = defaultdict(list)
        for idx, sample in enumerate(dataset.samples):
            for cat_id in sample['category_ids']:
                self.image_to_cats[idx].append(cat_id)
        
        print(f"EpisodeSampler initialized:")
        print(f"  Available categories: {len(self.available_cats)}")
        print(f"  n_way: {n_way}")
        print(f"  k_shot: {k_shot}")
        print(f"  n_query: {n_query}")
        print(f"   Cross-category deduplication: ENABLED")
    
    def sample_episode(self) -> Dict:
        """Build an episode with support and query images, labels, and class IDs."""
        episode_cats = random.sample(self.available_cats, self.n_way)
        
        support_samples = []
        query_samples = []
        
        used_support_indices = set()
        used_query_indices = set()
        for cat_id in episode_cats:
            cat_indices = self.cat_to_indices[cat_id]
            
            available_for_support = [idx for idx in cat_indices 
                                    if idx not in used_query_indices]
            available_for_query = [idx for idx in cat_indices 
                                 if idx not in used_support_indices]
            
            needed_support = self.k_shot
            needed_query = self.n_query
            
            if len(available_for_support) < needed_support or len(available_for_query) < needed_query:
                print(f"  Warning: Category {cat_id} has insufficient samples after deduplication")
                print(f"  Available for support: {len(available_for_support)}, needed: {needed_support}")
                print(f"  Available for query: {len(available_for_query)}, needed: {needed_query}")
                if len(cat_indices) < self.k_shot + self.n_query:
                    selected_indices = random.choices(cat_indices, k=self.k_shot + self.n_query)
                else:
                    selected_indices = random.sample(cat_indices, k=self.k_shot + self.n_query)
            else:
                support_indices = random.sample(available_for_support, needed_support)
                
                available_for_query_filtered = [idx for idx in available_for_query 
                                              if idx not in support_indices]
                
                if len(available_for_query_filtered) < needed_query:
                    remaining = needed_query - len(available_for_query_filtered)
                    query_indices = available_for_query_filtered + random.choices(
                        [idx for idx in cat_indices if idx not in available_for_query_filtered],
                        k=remaining
                    )
                else:
                    query_indices = random.sample(available_for_query_filtered, needed_query)
                
                selected_indices = support_indices + query_indices
            
            support_indices = selected_indices[:self.k_shot]
            query_indices = selected_indices[self.k_shot:self.k_shot + self.n_query]
            
            used_support_indices.update(support_indices)
            used_query_indices.update(query_indices)
            
            support_samples.extend(support_indices)
            query_samples.extend(query_indices)
        
        support_data = [self.dataset[idx] for idx in support_samples]
        query_data = [self.dataset[idx] for idx in query_samples]
        
        support_images = torch.stack([item['image'] for item in support_data])
        query_images = torch.stack([item['image'] for item in query_data])
        
        support_labels = [item['labels'].tolist() for item in support_data]
        query_labels = [item['labels'].tolist() for item in query_data]
        
        support_set = set(support_samples)
        query_set = set(query_samples)
        overlap = support_set.intersection(query_set)
        if overlap:
            print(f" ERROR: Found {len(overlap)} overlapping images between support and query sets!")
            print(f"  Overlapping indices: {overlap}")
        else:
            pass
        
        return {
            'support_images': support_images,
            'support_labels': support_labels,
            'query_images': query_images,
            'query_labels': query_labels,
            'episode_classes': episode_cats
        }

print("\nTesting EpisodeSampler (with cross-category deduplication)...")
test_dataset = MultiLabelCOCODataset(
    samples=val_samples,
    train_image_dir=TRAIN2014_DIR,
    val_image_dir=VAL2014_DIR,
    transform=test_transform,
    cat_id_to_name=cat_id_to_name
)

test_sampler = EpisodeSampler(
    dataset=test_dataset,
    n_way=5,
    k_shot=2,
    n_query=2
)

episode = test_sampler.sample_episode()
print(f"\nEpisode sampled:")
print(f"  Support images shape: {episode['support_images'].shape}")
print(f"  Query images shape: {episode['query_images'].shape}")
print(f"  Episode classes: {episode['episode_classes']}")
print(f"\nFirst support sample labels: {episode['support_labels'][0]}")
print(f" EpisodeSampler with deduplication test passed!")

## 3.8 Define the LC-Protonets Model


In [ ]:
class LCProtonets(nn.Module):
    """Classify query images using prototypes formed from support-label combinations."""
    
    def __init__(self, feature_extractor, distance_metric='euclidean', init_bias=2.0):
        super(LCProtonets, self).__init__()
        
        self.feature_extractor = feature_extractor
        self.distance_metric = distance_metric
        
        self.bias = nn.Parameter(torch.tensor(init_bias))
        
        print(f"LC-Protonets initialized:")
        print(f"  Distance metric: {distance_metric}")
        print(f"  Learnable bias: {self.bias.item():.4f} (initial={init_bias})")
        print(f"  F1 configuration: bias init={init_bias}")
    
    def compute_prototypes(
        self,
        support_features: torch.Tensor,
        support_labels: List[List[int]]
    ) -> Dict[Tuple[int], torch.Tensor]:
        """Average support embeddings for each observed label combination."""
        label_combinations = [tuple(sorted(labels)) for labels in support_labels]
        
        combination_to_features = defaultdict(list)
        for features, combo in zip(support_features, label_combinations):
            combination_to_features[combo].append(features)
        
        prototypes = {}
        for combo, feature_list in combination_to_features.items():
            features_stack = torch.stack(feature_list)  # (n_samples, feature_dim)
            prototype = features_stack.mean(dim=0)  # (feature_dim,)
            prototypes[combo] = prototype
        
        return prototypes
    
    def compute_distances(
        self,
        query_features: torch.Tensor,
        prototypes: Dict[Tuple[int], torch.Tensor]
    ) -> Dict[Tuple[int], torch.Tensor]:
        """Compute distances from query embeddings to each prototype."""
        distances = {}
        
        for combo, prototype in prototypes.items():
            if self.distance_metric == 'euclidean':
                dist = torch.norm(query_features - prototype.unsqueeze(0), dim=1)
            elif self.distance_metric == 'cosine':
                query_norm = F.normalize(query_features, p=2, dim=1)
                prototype_norm = F.normalize(prototype.unsqueeze(0), p=2, dim=1)
                similarity = F.cosine_similarity(query_norm, prototype_norm, dim=1)
                dist = 1 - similarity
            else:
                raise ValueError(f"Unknown distance metric: {self.distance_metric}")
            
            distances[combo] = dist
        
        return distances
    
    def aggregate_to_labels(
        self,
        distances: Dict[Tuple[int], torch.Tensor],
        episode_classes: List[int],
        aggregation='max'
    ) -> torch.Tensor:
        """Aggregate combination scores into one logit per episode class."""
        n_query = list(distances.values())[0].shape[0]
        n_classes = len(episode_classes)
        device = list(distances.values())[0].device
        
        if aggregation == 'max':
            label_scores = torch.full((n_query, n_classes), float('inf'), device=device)
        else:  # min or mean
            label_scores = torch.zeros((n_query, n_classes), device=device)
        
        temp_scores = []
        
        for combo, dist in distances.items():
            for cat_id in combo:
                if cat_id in episode_classes:
                    class_idx = episode_classes.index(cat_id)
                    temp_scores.append((class_idx, dist))
        
        if aggregation == 'max':
            for class_idx, dist in temp_scores:
                label_scores[:, class_idx] = torch.where(
                    dist < label_scores[:, class_idx],
                    dist,
                    label_scores[:, class_idx]
                )
        elif aggregation == 'min':
            for class_idx, dist in temp_scores:
                label_scores[:, class_idx] = torch.where(
                    dist > label_scores[:, class_idx],
                    dist,
                    label_scores[:, class_idx]
                )
        elif aggregation == 'mean':
            counts = torch.zeros(n_classes, device=device)
            for class_idx, dist in temp_scores:
                label_scores[:, class_idx] += dist
                counts[class_idx] += 1
            counts = torch.where(counts > 0, counts, torch.ones_like(counts))
            label_scores = label_scores / counts.unsqueeze(0)
        
        label_logits = -label_scores + self.bias
        
        return label_logits
    
    def forward(self, episode):
        """Compute query logits and label-combination prototypes for an episode."""
        support_features = self.feature_extractor(episode['support_images'])
        query_features = self.feature_extractor(episode['query_images'])
        
        prototypes = self.compute_prototypes(support_features, episode['support_labels'])
        
        distances = self.compute_distances(query_features, prototypes)
        
        predictions = self.aggregate_to_labels(
            distances,
            episode['episode_classes'],
            aggregation='max'
        )
        
        return predictions, prototypes

print("\nTesting LC-Protonets with configured bias initialization...")
lc_protonets = LCProtonets(
    feature_extractor=feature_extractor,
    distance_metric='euclidean',
    init_bias=2.0
)

test_episode = {
    'support_images': torch.randn(10, 3, 224, 224).to(DEVICE),
    'support_labels': [[1], [2], [1, 2], [3], [1, 3], [4], [2, 4], [5], [1, 2, 5], [3, 4, 5]],
    'query_images': torch.randn(20, 3, 224, 224).to(DEVICE),
    'query_labels': [[1], [2], [3], [4], [5]],
    'episode_classes': [1, 2, 3, 4, 5]
}

predictions, prototypes = lc_protonets(test_episode)

print(f"\nPredictions shape: {predictions.shape}")
print(f"Predictions range: [{predictions.min():.4f}, {predictions.max():.4f}]")
print(f"Predictions mean: {predictions.mean():.4f}")
print(f"Initial bias: {lc_protonets.bias.item():.4f}")

probs = torch.sigmoid(predictions)
print(f"\n Probability range: [{probs.min():.4f}, {probs.max():.4f}]")
print(f" Probability mean: {probs.mean():.4f}")
print(f" Probability >0.5: {(probs > 0.5).sum().item()} / {probs.numel()}")
print(f" Probability >0.7: {(probs > 0.7).sum().item()} / {probs.numel()}")
print(f" Probability >0.9: {(probs > 0.9).sum().item()} / {probs.numel()}")

print(f"\nNumber of prototypes: {len(prototypes)}")
print(f" LC-Protonets test passed! Model now starts with bias=2.0.")

## 3.9 Define the Loss Function


In [ ]:
class MultiLabelLoss(nn.Module):
    """Compute a multi-label classification loss over episode classes."""
    
    def __init__(self, reduction='mean'):
        super(MultiLabelLoss, self).__init__()
        self.reduction = reduction
    
    def forward(self, predictions, labels_list, episode_classes):
        """Build binary targets for the episode classes and compute the loss."""
        n_query, n_classes = predictions.shape
        
        targets = torch.zeros(n_query, n_classes).to(predictions.device)
        
        for i, labels in enumerate(labels_list):
            for label in labels:
                if label in episode_classes:
                    class_idx = episode_classes.index(label)
                    targets[i, class_idx] = 1
        
        loss = F.binary_cross_entropy_with_logits(predictions, targets, reduction='none')
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

print("Testing MultiLabelLoss...")
criterion = MultiLabelLoss()

test_predictions = torch.randn(10, 5)
test_labels = [[1], [2], [1, 2], [3], [1, 3], [4], [2, 4], [5], [1, 2, 5], [3, 4, 5]]
test_classes = [1, 2, 3, 4, 5]

loss = criterion(test_predictions, test_labels, test_classes)
print(f"Loss value: {loss.item():.4f}")
print(f"\nMultiLabelLoss test passed!")

## 3.10 Implement Training and Evaluation Functions


In [ ]:
def train_episode(
    model: LCProtonets,
    episode: Dict,
    criterion: MultiLabelLoss,
    optimizer: torch.optim.Optimizer,
    device: torch.device
) -> Dict:
    """Run one training episode and update the model parameters."""
    model.train()
    
    episode = {
        'support_images': episode['support_images'].to(device),
        'support_labels': episode['support_labels'],
        'query_images': episode['query_images'].to(device),
        'query_labels': episode['query_labels'],
        'episode_classes': episode['episode_classes']
    }
    
    predictions, prototypes = model(episode)
    
    loss = criterion(predictions, episode['query_labels'], episode['episode_classes'])
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    with torch.no_grad():
        pred_labels = (torch.sigmoid(predictions) > 0.5).float()
        
        n_query, n_classes = predictions.shape
        target_tensor = torch.zeros(n_query, n_classes).to(device)
        
        for i, labels in enumerate(episode['query_labels']):
            for label in labels:
                if label in episode['episode_classes']:
                    class_idx = episode['episode_classes'].index(label)
                    target_tensor[i, class_idx] = 1
        
        accuracy = (pred_labels == target_tensor).float().mean()
    
    return {
        'loss': loss.item(),
        'accuracy': accuracy.item(),
        'num_prototypes': len(prototypes)
    }


@torch.no_grad()
def evaluate_episode(
    model: LCProtonets,
    episode: Dict,
    criterion: MultiLabelLoss,
    device: torch.device
) -> Dict:
    """Evaluate the model on one episode and return predictions and targets."""
    model.eval()
    
    episode = {
        'support_images': episode['support_images'].to(device),
        'support_labels': episode['support_labels'],
        'query_images': episode['query_images'].to(device),
        'query_labels': episode['query_labels'],
        'episode_classes': episode['episode_classes']
    }
    
    predictions, prototypes = model(episode)
    
    loss = criterion(predictions, episode['query_labels'], episode['episode_classes'])
    
    pred_labels = (torch.sigmoid(predictions) > 0.5).float()
    
    n_query, n_classes = predictions.shape
    target_tensor = torch.zeros(n_query, n_classes).to(device)
    
    for i, labels in enumerate(episode['query_labels']):
        for label in labels:
            if label in episode['episode_classes']:
                class_idx = episode['episode_classes'].index(label)
                target_tensor[i, class_idx] = 1
    
    accuracy = (pred_labels == target_tensor).float().mean()
    
    return {
        'loss': loss.item(),
        'accuracy': accuracy.item(),
        'predictions': predictions.cpu(),        'targets': target_tensor.cpu(),
        'num_prototypes': len(prototypes)
    }

print(" Train and evaluate functions defined successfully! ")

In [ ]:
import torch
import numpy as np
from sklearn.metrics import average_precision_score, f1_score

def quick_diagnose(predictions, targets):
    """Quickly diagnose why F1 is zero."""
    pred_np = predictions.numpy() if torch.is_tensor(predictions) else predictions
    targ_np = targets.numpy() if torch.is_tensor(targets) else targets

    print("\n" + "="*70)
    print(" F1 Diagnostic Report")
    print("="*70)

    print(f"\nPrediction statistics:")
    print(f"  Range: [{pred_np.min():.4f}, {pred_np.max():.4f}]")
    print(f"  Mean: {pred_np.mean():.4f}")
    print(f"  Target positive rate: {targ_np.mean():.4f}")
    print(f"\nF1 scores at different thresholds:")
    for thresh in [0.1, 0.2, 0.3, 0.4, 0.5]:
        pred_bin = (pred_np > thresh).astype(int)
        tp = ((pred_bin == 1) & (targ_np == 1)).sum()
        fp = ((pred_bin == 1) & (targ_np == 0)).sum()
        fn = ((pred_bin == 0) & (targ_np == 1)).sum()

        pos_count = pred_bin.sum()
        if tp > 0:
            f1 = 2 * tp / (2 * tp + fp + fn)
        else:
            f1 = 0.0

        print(f"  threshold={thresh:.1f}: positive predictions={pos_count:4d}, F1={f1:.4f}")
    try:
        map_micro = average_precision_score(targ_np, pred_np, average='micro')
        print(f"\nAP (micro): {map_micro:.4f}")
    except:
        pass
    
    print(f"\n{'='*70}")
    print(" Per-Class Analysis (why Macro is approximately Micro)")
    print(f"{'='*70}")
    
    n_classes = targ_np.shape[1]
    per_class_ap = []
    per_class_f1 = []
    
    for c in range(n_classes):
        try:
            ap = average_precision_score(targ_np[:, c], pred_np[:, c])
            per_class_ap.append(ap)
        except:
            per_class_ap.append(0.0)
        
        pred_bin_c = (pred_np[:, c] > 0.5).astype(int)
        tp_c = ((pred_bin_c == 1) & (targ_np[:, c] == 1)).sum()
        fp_c = ((pred_bin_c == 1) & (targ_np[:, c] == 0)).sum()
        fn_c = ((pred_bin_c == 0) & (targ_np[:, c] == 1)).sum()
        
        if tp_c > 0:
            f1_c = 2 * tp_c / (2 * tp_c + fp_c + fn_c)
        else:
            f1_c = 0.0
        per_class_f1.append(f1_c)
    
    per_class_ap = np.array(per_class_ap)
    per_class_f1 = np.array(per_class_f1)
    
    print(f"\nAP per class:")
    print(f"  Mean: {np.mean(per_class_ap):.4f}")
    print(f"  Std:  {np.std(per_class_ap):.4f}")
    print(f"  Min:  {np.min(per_class_ap):.4f}")
    print(f"  Max:  {np.max(per_class_ap):.4f}")
    
    print(f"\nF1 per class:")
    print(f"  Mean: {np.mean(per_class_f1):.4f}")
    print(f"  Std:  {np.std(per_class_f1):.4f}")
    print(f"  Min:  {np.min(per_class_f1):.4f}")
    print(f"  Max:  {np.max(per_class_f1):.4f}")
    
    print(f"\nPer-class AP and F1 details:")
    for c in range(n_classes):
        print(f"  Class {c:2d}: AP={per_class_ap[c]:.4f}, F1={per_class_f1[c]:.4f}")
    
    print(f"\nAnalysis:")
    if np.std(per_class_ap) < 0.05 and np.std(per_class_f1) < 0.08:
        print(f"  Standard deviations are small (AP std={np.std(per_class_ap):.4f}, F1 std={np.std(per_class_f1):.4f})")
        print(f"  Class performance is consistent, so Macro is approximately Micro")
    else:
        print(f"  Standard deviations are large (AP std={np.std(per_class_ap):.4f}, F1 std={np.std(per_class_f1):.4f})")
        print(f"  Class performance varies, so Macro and Micro should differ")    
    print("="*70)

print(" Diagnostic function loaded (including per-class analysis)")

## 3.11 Implement the Training Loop


In [ ]:
def train_lc_protonets(
    train_dataset: MultiLabelCOCODataset,
    val_dataset: MultiLabelCOCODataset,
    n_way: int,
    k_shot: int,
    n_query: int,
    n_episodes: int,
    val_intervals: int,
    device: torch.device,
    save_path: str = None,
    distance_metric: str = 'euclidean',
    feature_dim: int = 2048,
    finetune: bool = True,
    learning_rate: float = 0.0001,
    gradient_accumulation_steps: int = 1,
    use_mixed_precision: bool = False,
    val_n_way: int = None,
    init_bias: float = 2.0,
    bias_lr_multiplier: int = 20
):
    """Train LC-Protonets and evaluate validation episodes at regular intervals."""
    feature_extractor = FeatureExtractor(
        pretrained=True, 
        feature_dim=feature_dim,
        finetune=finetune
    ).to(device)
    
    model = LCProtonets(feature_extractor, distance_metric=distance_metric, init_bias=init_bias)
    
    if finetune:
        optimizer = torch.optim.Adam([
            {'params': model.feature_extractor.features.parameters(), 'lr': learning_rate * 0.1},
            {'params': model.feature_extractor.projection.parameters(), 'lr': learning_rate},
            {'params': model.bias, 'lr': learning_rate * bias_lr_multiplier}        ])
    else:
        optimizer = torch.optim.Adam([
            {'params': model.feature_extractor.projection.parameters(), 'lr': learning_rate},
            {'params': model.bias, 'lr': learning_rate * bias_lr_multiplier}        ])
    
    scaler = None
    if use_mixed_precision:
        from torch.cuda.amp import GradScaler
        scaler = GradScaler()
    
    criterion = MultiLabelLoss()
    
    train_sampler = EpisodeSampler(train_dataset, n_way, k_shot, n_query)
    
    val_sampler_obj = EpisodeSampler(val_dataset, min(n_way, 12), k_shot, n_query)
    val_n_way_actual = min(val_n_way or n_way, 12)
    print(f"\n{'='*70}")
    print(f"TRAINING LC-PROTONETS")
    print(f"{'='*70}")
    print(f"Configuration:")
    print(f"  Train n_way: {n_way}, Val n_way: {val_n_way_actual}")
    print(f"  k_shot: {k_shot}, n_query: {n_query}")
    print(f"  n_episodes: {n_episodes}")
    print(f"  val_intervals: {val_intervals}")
    print(f"  distance_metric: {distance_metric}")
    print(f"  feature_dim: {feature_dim}")
    print(f"  finetune: {finetune}")
    print(f"  learning_rate: {learning_rate}")
    print(f"  gradient_accumulation_steps: {gradient_accumulation_steps}")
    print(f"  use_mixed_precision: {use_mixed_precision}")
    print(f"\n F1 Configuration:")
    print(f"  init_bias: {init_bias}")
    print(f"  bias learning rate: {learning_rate * bias_lr_multiplier:.6f} ({bias_lr_multiplier}x base lr)")
    print(f"{'='*70}\n")
    
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'val_map_macro': [],
        'val_map_micro': [],
        'val_f1_macro': [],
        'val_f1_micro': []
    }
    
    best_val_acc = 0.0
    
    accumulation_counter = 0
    
    for episode_idx in tqdm(range(n_episodes), desc="Training"):
        model.train()
        
        train_ep = train_sampler.sample_episode()
        
        train_ep = {
            'support_images': train_ep['support_images'].to(device),
            'support_labels': train_ep['support_labels'],
            'query_images': train_ep['query_images'].to(device),
            'query_labels': train_ep['query_labels'],
            'episode_classes': train_ep['episode_classes']
        }
        
        if use_mixed_precision:
            with torch.amp.autocast('cuda'):
                predictions, prototypes = model(train_ep)
                loss = criterion(predictions, train_ep['query_labels'], train_ep['episode_classes'])
                loss = loss / gradient_accumulation_steps
        else:
            predictions, prototypes = model(train_ep)
            loss = criterion(predictions, train_ep['query_labels'], train_ep['episode_classes'])
            loss = loss / gradient_accumulation_steps
        if use_mixed_precision:
            scaler.scale(loss).backward()
        else:
            loss.backward()
        
        accumulation_counter += 1
        
        if accumulation_counter % gradient_accumulation_steps == 0:
            if use_mixed_precision:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad()
        
        with torch.no_grad():
            pred_labels = (torch.sigmoid(predictions) > 0.5).float()
            
            n_query, n_classes = predictions.shape
            target_tensor = torch.zeros(n_query, n_classes).to(device)
            
            for i, labels in enumerate(train_ep['query_labels']):
                for label in labels:
                    if label in train_ep['episode_classes']:
                        class_idx = train_ep['episode_classes'].index(label)
                        target_tensor[i, class_idx] = 1
            
            accuracy = (pred_labels == target_tensor).float().mean()
        
        history['train_loss'].append(loss.item() * gradient_accumulation_steps)
        history['train_acc'].append(accuracy.item())

        completed_episode = episode_idx + 1
        if completed_episode == 1 or completed_episode % 10 == 0 or completed_episode == n_episodes:
            update_run_progress(
                'LC-Protonets training',
                completed_episode,
                n_episodes,
                loss=float(history['train_loss'][-1]),
                accuracy=float(history['train_acc'][-1])
            )
        
        if (episode_idx + 1) % val_intervals == 0:
            update_run_progress(
                'LC-Protonets validation',
                episode_idx + 1,
                n_episodes,
                validation_episodes=50
            )
            model.eval()
            val_losses = []
            val_accs = []
            num_val_episodes = 50
            
            all_predictions_list = []
            all_targets_list = []
            
            with torch.no_grad():
                for _ in range(num_val_episodes):
                    val_ep = val_sampler_obj.sample_episode()
                    val_ep = {
                        'support_images': val_ep['support_images'].to(device),
                        'support_labels': val_ep['support_labels'],
                        'query_images': val_ep['query_images'].to(device),
                        'query_labels': val_ep['query_labels'],
                        'episode_classes': val_ep['episode_classes']
                    }
                    
                    if use_mixed_precision:
                        with torch.amp.autocast('cuda'):
                            val_pred, val_prot = model(val_ep)
                            val_loss = criterion(val_pred, val_ep['query_labels'], val_ep['episode_classes'])
                    else:
                        val_pred, val_prot = model(val_ep)
                        val_loss = criterion(val_pred, val_ep['query_labels'], val_ep['episode_classes'])
                    
                    val_losses.append(val_loss.item())
                    
                    val_pred_labels = (torch.sigmoid(val_pred) > 0.5).float()
                    val_n_query, val_n_classes = val_pred.shape
                    val_target = torch.zeros(val_n_query, val_n_classes).to(device)
                    
                    for i, labels in enumerate(val_ep['query_labels']):
                        for label in labels:
                            if label in val_ep['episode_classes']:
                                class_idx = val_ep['episode_classes'].index(label)
                                val_target[i, class_idx] = 1
                    
                    val_acc = (val_pred_labels == val_target).float().mean()
                    val_accs.append(val_acc.item())
                    
                    all_predictions_list.append(torch.sigmoid(val_pred).cpu())
                    all_targets_list.append(val_target.cpu())
            
            all_predictions = torch.cat(all_predictions_list, dim=0)  # (total_samples, n_classes)
            all_targets = torch.cat(all_targets_list, dim=0)  # (total_samples, n_classes)
            
            avg_val_loss = np.mean(val_losses)
            avg_val_acc = np.mean(val_accs)
            
            metrics = compute_multilabel_metrics(all_predictions, all_targets)
            
            if episode_idx + 1 == val_intervals or episode_idx + 1 == n_episodes:
                print(f"\n{'='*70}")
                print(f"Detailed diagnostics - Episode {episode_idx + 1}/{n_episodes}")
                print(f"{'='*70}")
                quick_diagnose(all_predictions, all_targets)

            history['val_loss'].append(avg_val_loss)
            history['val_acc'].append(avg_val_acc)
            history['val_map_macro'].append(metrics['map_macro'])
            history['val_map_micro'].append(metrics['map_micro'])
            history['val_f1_macro'].append(metrics['f1_macro'])
            history['val_f1_micro'].append(metrics['f1_micro'])
            
            print(f"\nEpisode {episode_idx + 1}/{n_episodes}:")
            print(f"  Train Loss: {loss.item() * gradient_accumulation_steps:.4f}, Train Acc: {accuracy.item():.4f}")
            print(f"  Val Loss: {avg_val_loss:.4f}, Val Acc: {avg_val_acc:.4f}")
            print(f"  Val AP (macro): {metrics['map_macro']:.4f}, Val AP (micro): {metrics['map_micro']:.4f}")
            print(f"  Val F1 (macro): {metrics['f1_macro']:.4f}, Val F1 (micro): {metrics['f1_micro']:.4f}")
            print(f"  Num Prototypes: {len(prototypes)}")
            print(f"   Current bias: {model.bias.item():.4f} (started at {init_bias:.2f}, lr={learning_rate * bias_lr_multiplier:.6f})")  # Display bias and learning rate            
            if torch.cuda.is_available():
                allocated = torch.cuda.memory_allocated(0) / 1024**3
                cached = torch.cuda.memory_reserved(0) / 1024**3
                print(f"  GPU Memory: {allocated:.2f} GB allocated, {cached:.2f} GB cached")
            
            # if metrics['map_micro'] > best_val_acc:
            #     best_val_acc = metrics['map_micro']
            #     if save_path:
            #         torch.save(model.state_dict(), save_path)
            #         print(f"   Saved best model (val mAP_micro: {best_val_acc:.4f})")
            
            if metrics['map_macro'] > best_val_acc:
                best_val_acc = metrics['map_macro']
                if save_path:
                    torch.save(model.state_dict(), save_path)
                    print(f"   Saved best model (val mAP_macro: {best_val_acc:.4f})")
            
            torch.cuda.empty_cache()
    
    print(f"\n{'='*70}")
    print(f"TRAINING COMPLETED")
    print(f"Best validation mAP (micro): {best_val_acc:.4f}")
    print(f"Final bias value: {model.bias.item():.4f} (started at {init_bias:.2f})")
    print(f"{'='*70}")
    
    return history, model


def compute_multilabel_metrics(predictions, targets, verbose=False):
    """Compute macro- and micro-averaged AP and F1 scores."""
    from sklearn.metrics import average_precision_score, f1_score, precision_recall_fscore_support
    
    predictions_np = predictions.numpy()
    targets_np = targets.numpy()
    
    if verbose:
        print(f"\n=== Debug Info ===")
        print(f"Predictions shape: {predictions_np.shape}")
        print(f"Targets shape: {targets_np.shape}")
        print(f"Predictions range: [{predictions_np.min():.4f}, {predictions_np.max():.4f}]")
        print(f"Targets sum (total positives): {targets_np.sum()}")
        print(f"Targets mean (positive ratio): {targets_np.mean():.4f}")
        
        pred_binary = (predictions_np > 0.5).astype(int)
        print(f"Predicted positives (threshold=0.5): {pred_binary.sum()}")
        print(f"Predicted positive ratio: {pred_binary.mean():.4f}")
        
        n_classes = targets_np.shape[1]
        print(f"\nPer-class statistics:")
        for c in range(min(n_classes, 5)):
            n_pos = targets_np[:, c].sum()
            n_pred_pos = pred_binary[:, c].sum()
            print(f"  Class {c}: {n_pos} positives, {n_pred_pos} predicted positives")
    
    try:
        map_macro = average_precision_score(targets_np, predictions_np, average='macro')
        
        map_micro = average_precision_score(targets_np, predictions_np, average='micro')
    except Exception as e:
        print(f"Error computing AP: {e}")
        map_macro = 0.0
        map_micro = 0.0
    
    pred_labels = (predictions_np > 0.5).astype(int)
    
    try:
        f1_macro = f1_score(targets_np, pred_labels, average='macro', zero_division=0)
        
        if verbose:
            precision_macro, recall_macro, f1_per_class, _ = precision_recall_fscore_support(
                targets_np, pred_labels, average=None, zero_division=0
            )
            print(f"\nPer-class F1 scores (first 5 classes):")
            for c in range(min(len(f1_per_class), 5)):
                print(f"  Class {c}: F1={f1_per_class[c]:.4f}")
            print(f"Macro F1 (mean of per-class): {f1_per_class.mean():.4f}")
            
            tp = ((pred_labels == 1) & (targets_np == 1)).sum()
            fp = ((pred_labels == 1) & (targets_np == 0)).sum()
            fn = ((pred_labels == 0) & (targets_np == 1)).sum()
            print(f"\nGlobal counts:")
            print(f"  TP: {tp}, FP: {fp}, FN: {fn}")
            if tp + fp > 0:
                precision = tp / (tp + fp)
                print(f"  Precision: {precision:.4f}")
            if tp + fn > 0:
                recall = tp / (tp + fn)
                print(f"  Recall: {recall:.4f}")
            if tp + fp + fn > 0:
                f1_manual = 2 * tp / (2 * tp + fp + fn)
                print(f"  F1 (manual): {f1_manual:.4f}")
    
    except Exception as e:
        print(f"Error computing F1: {e}")
        f1_macro = 0.0
        f1_micro = 0.0
    
    return {
        'map_macro': map_macro,
        'map_micro': map_micro,
        'f1_macro': f1_macro,
        'f1_micro': f1_micro
    }

print("Training and metric functions are ready.")

## 3.12 Prepare Datasets


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = MultiLabelCOCODataset(
    samples=base_samples,
    train_image_dir=TRAIN2014_DIR,
    val_image_dir=VAL2014_DIR,
    transform=train_transform,
    cat_id_to_name=cat_id_to_name
)

val_dataset = MultiLabelCOCODataset(
    samples=val_samples,
    train_image_dir=TRAIN2014_DIR,
    val_image_dir=VAL2014_DIR,
    transform=val_transform,
    cat_id_to_name=cat_id_to_name
)

print(f"\nDatasets prepared:")
print(f"  Train dataset: {len(train_dataset)} samples")
print(f"  Val dataset: {len(val_dataset)} samples")

In [ ]:
novel_dataset = MultiLabelCOCODataset(
    samples=novel_samples,
    train_image_dir=TRAIN2014_DIR,
    val_image_dir=VAL2014_DIR,
    transform=val_transform,    cat_id_to_name=cat_id_to_name
)

print(f"\n Novel/Test dataset prepared:")
print(f"  Novel dataset: {len(novel_dataset)} samples (16 classes)")
print(f"\n Dataset Summary:")
print(f"  Train: {len(train_dataset)} samples (52 classes)")
print(f"  Val:   {len(val_dataset)} samples (12 classes)")
print(f"  Test:  {len(novel_dataset)} samples (16 classes)")

## 3.13 Configure Training Parameters


In [ ]:
TRAIN_N_WAY_FULL = 52
VAL_N_WAY_FULL = 12
NOVEL_N_WAY_FULL = 16
N_WAY = 52
K_SHOT = 5
N_QUERY = 4
N_EPISODES = 3000
VAL_INTERVALS = 200
FEATURE_DIM = 2048
FINETUNE = True
LEARNING_RATE = 0.0001
DISTANCE_METRIC = 'euclidean'
INIT_BIAS = 14.7
BIAS_LR_MULTIPLIER = 30
GRADIENT_ACCUMULATION_STEPS = 4
USE_MIXED_PRECISION = True
model_name = f"lc_protonets_{N_WAY}way_{K_SHOT}shot_dim{FEATURE_DIM}_finetune{FINETUNE}_{DISTANCE_METRIC}_bias{INIT_BIAS}.pth"
MODEL_SAVE_PATH = os.path.join(CHECKPOINT_DIR, model_name)

print(f"\n{'='*70}")
print(f"TRAINING CONFIGURATION")
print(f"{'='*70}")
print(f"Dataset Information:")
print(f"  Train set: {TRAIN_N_WAY_FULL} classes (base)")
print(f"  Val set: {VAL_N_WAY_FULL} classes (validation)")
print(f"  Novel set: {NOVEL_N_WAY_FULL} classes (novel/test)")
print(f"\nTraining Configuration:")
print(f"  Train n_way: {N_WAY} (sampling from {TRAIN_N_WAY_FULL} classes)")
print(f"  Val n_way: auto (using all {VAL_N_WAY_FULL} classes)")
print(f"  k_shot: {K_SHOT}")
print(f"  n_query: {N_QUERY}")
print(f"  n_episodes: {N_EPISODES}")
print(f"  val_intervals: {VAL_INTERVALS}")
print(f"  feature_dim: {FEATURE_DIM}")
print(f"  finetune: {FINETUNE}")
print(f"  learning_rate: {LEARNING_RATE}")
print(f"  distance_metric: {DISTANCE_METRIC}")
print(f"  gradient_accumulation_steps: {GRADIENT_ACCUMULATION_STEPS}")
print(f"  mixed_precision: {USE_MIXED_PRECISION}")
print(f"\nBias Configuration:")
print(f"  init_bias: {INIT_BIAS}")
print(f"  bias learning rate: {LEARNING_RATE * BIAS_LR_MULTIPLIER:.6f} ({BIAS_LR_MULTIPLIER}x base lr)")
print(f"Model save path: {MODEL_SAVE_PATH}")
print(f"{'='*70}\n")

if torch.cuda.is_available():
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    allocated_memory = torch.cuda.memory_allocated(0) / 1024**3
    print(f"GPU Memory Status:")
    print(f"  Total: {total_memory:.2f} GB")
    print(f"  Allocated: {allocated_memory:.2f} GB")
    print(f"  Available: {total_memory - allocated_memory:.2f} GB")
    
    estimated_memory = (N_WAY * (K_SHOT + N_QUERY) * 224 * 224 * 3 * 4 / 1024**3)
    print(f"\n  Estimated memory per episode: ~{estimated_memory:.2f} GB")
    
    print(f"\n{'='*70}")
    print(f"GPU MEMORY RECOMMENDATIONS")
    print(f"{'='*70}")
    
    if total_memory < 12:
        print(f"  WARNING: GPU memory is limited ({total_memory:.1f}GB)")
        print(f"  Recommended settings: N_WAY=8, K_SHOT=5, N_QUERY=2")
        print(f"  Alternatively, set FINETUNE=False")
    elif total_memory < 20:
        print(f" Note: GPU memory is moderate ({total_memory:.1f}GB)")
        print(f"  The current configuration (N_WAY={N_WAY}) should work")
        print(f"  If out-of-memory errors persist, set N_WAY=12")
    elif total_memory < 32:
        print(f" GPU memory is good ({total_memory:.1f}GB)")
        print(f"  The current configuration (N_WAY={N_WAY}) should work well")
        print(f"  After training, consider trying N_WAY=32 for better performance")
    else:
        print(f" GPU memory is sufficient ({total_memory:.1f}GB)")
        print(f"  The full configuration is available: N_WAY={TRAIN_N_WAY_FULL}")
        print(f"  Set N_WAY={TRAIN_N_WAY_FULL} for best performance")
    print(f"{'='*70}\n")

print(f"  IMPORTANT: Training will take a long time!")
print(f"Estimated time: {N_EPISODES * 2 / 60:.1f} minutes (assuming 2 seconds per episode)")
print("\nTo start training, run the cell below.")
print("You can also modify the parameters above before training.")

print(f"\n{'='*70}")
print(f"MEMORY OPTIMIZATION OPTIONS")
print(f"{'='*70}")
print(f"If you encounter OOM (Out Of Memory):")
print(f"  1. Reduce N_WAY: 52 to 32 to 16 to 12 to 8 to 5")
print(f"  2. Reduce K_SHOT: 5 to 3 to 1")
print(f"  3. Reduce N_QUERY: 4 to 2")
print(f"  4. Set FINETUNE = False (freezes ResNet)")
print(f"  5. Set GRADIENT_ACCUMULATION_STEPS = 2 or 4")
print(f"  6. Set USE_MIXED_PRECISION = True")
print(f"{'='*70}\n")

print(f"{'='*70}")
print(f"DATASET SPLIT INFORMATION")
print(f"{'='*70}")
print(f"Training Strategy:")
print(f"  - Each episode samples {N_WAY} classes from {TRAIN_N_WAY_FULL} training classes")
print(f"  - Validation uses all {VAL_N_WAY_FULL} classes")
print(f"  - Testing uses all {NOVEL_N_WAY_FULL} novel classes")
print(f"\nRationale:")
print(f"  - Training on random subsets improves generalization")
print(f"  - Validation/Test on all classes provides comprehensive evaluation")
print(f"{'='*70}\n")

## 3.14 Train the Model


In [ ]:
history, trained_model = train_lc_protonets(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    n_way=N_WAY,
    k_shot=K_SHOT,
    n_query=N_QUERY,
    n_episodes=N_EPISODES,
    val_intervals=VAL_INTERVALS,
    device=DEVICE,
    save_path=MODEL_SAVE_PATH,
    distance_metric=DISTANCE_METRIC,
    feature_dim=FEATURE_DIM,
    finetune=FINETUNE,
    learning_rate=LEARNING_RATE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    use_mixed_precision=USE_MIXED_PRECISION,
    init_bias=INIT_BIAS,    bias_lr_multiplier=BIAS_LR_MULTIPLIER
)

print("\n Training completed!")
print(f"Model saved to: {MODEL_SAVE_PATH}")
print(f"Bias configuration: init_bias={INIT_BIAS}, bias_lr={BIAS_LR_MULTIPLIER}x")

## 3.15 Define Full Evaluation and Visualization


In [ ]:
@torch.no_grad()
def evaluate_full(
    model: LCProtonets,
    dataset: MultiLabelCOCODataset,
    n_way: int,
    k_shot: int,
    n_query: int,
    n_episodes: int,
    device: torch.device
):
    """Evaluate LC-Protonets over multiple few-shot episodes."""
    model.eval()

    sampler = EpisodeSampler(dataset, n_way, k_shot, n_query)
    criterion = MultiLabelLoss()

    all_predictions = []
    all_targets = []
    all_losses = []
    all_accs = []

    print(f"\nEvaluating on {n_episodes} episodes...")

    for i in tqdm(range(n_episodes), desc="Evaluating"):
        episode = sampler.sample_episode()
        metrics = evaluate_episode(model, episode, criterion, device)

        all_predictions.append(torch.sigmoid(metrics['predictions']))
        all_targets.append(metrics['targets'])
        all_losses.append(metrics['loss'])
        all_accs.append(metrics['accuracy'])

    all_predictions = torch.cat(all_predictions, dim=0)
    all_targets = torch.cat(all_targets, dim=0)

    avg_loss = np.mean(all_losses)
    avg_acc = np.mean(all_accs)

    metrics = compute_multilabel_metrics(all_predictions, all_targets, verbose=False)

    per_class_acc = []
    n_classes = all_targets.shape[1]

    for c in range(n_classes):
        class_preds = (all_predictions[:, c] > 0.5).float()
        class_targets = all_targets[:, c]
        class_acc = (class_preds == class_targets).float().mean()
        per_class_acc.append(class_acc.item())

    print(f"\n{'='*70}")
    print(f"EVALUATION RESULTS")
    print(f"{'='*70}")
    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Average Accuracy: {avg_acc:.4f}")
    print(f"Per-class Accuracy: {np.mean(per_class_acc):.4f} +/- {np.std(per_class_acc):.4f}")
    print(f"\nFour Core Metrics:")
    print(f"  Macro AP (mAP):       {metrics['map_macro']:.4f}")
    print(f"  Micro AP:             {metrics['map_micro']:.4f}")
    print(f"  Macro F1 (mF1):       {metrics['f1_macro']:.4f}")
    print(f"  Micro F1:             {metrics['f1_micro']:.4f}")
    print(f"{'='*70}")
    return {
        'avg_loss': avg_loss,
        'avg_acc': avg_acc,
        'per_class_acc': per_class_acc,
        'all_predictions': all_predictions,
        'all_targets': all_targets,
        'map_macro': metrics['map_macro'],
        'map_micro': metrics['map_micro'],
        'f1_macro': metrics['f1_macro'],
        'f1_micro': metrics['f1_micro']
    }


def plot_evaluation_metrics(results, dataset_name="Dataset", save_dir=None):
    """Plot aggregate and per-class evaluation metrics."""
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.metrics import f1_score, average_precision_score
    import numpy as np
    import os

    plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
    predictions = results['all_predictions'].numpy()
    targets = results['all_targets'].numpy()

    sns.set_style("whitegrid")
    plt.rcParams['figure.figsize'] = (20, 12)

    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle(f'{dataset_name} - Evaluation Results Visualization', fontsize=16, fontweight='bold')

    ax1 = axes[0, 0]
    metrics_names = ['mAP\n(Macro Avg)', 'microAP\n(Micro Avg)', 'mF1\n(Macro Avg)', 'microF1\n(Micro Avg)']
    metrics_values = [
        results['map_macro'],
        results['map_micro'],
        results['f1_macro'],
        results['f1_micro']
    ]
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
    bars = ax1.bar(metrics_names, metrics_values, color=colors, alpha=0.8, edgecolor='black')
    ax1.set_ylabel('Score', fontsize=11, fontweight='bold')
    ax1.set_title('Four Core Metrics Comparison', fontsize=12, fontweight='bold')
    ax1.set_ylim([0, 1])
    ax1.grid(axis='y', alpha=0.3)

    for bar, value in zip(bars, metrics_values):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{value:.4f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax2 = axes[0, 1]
    per_class_acc = results['per_class_acc']
    bp = ax2.boxplot([per_class_acc], tick_labels=['Accuracy'], patch_artist=True,
                     boxprops=dict(facecolor='lightblue', alpha=0.7),
                     medianprops=dict(color='red', linewidth=2))
    ax2.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
    ax2.set_title(f'Per-class Accuracy Distribution\n(Mean: {np.mean(per_class_acc):.4f} +/- {np.std(per_class_acc):.4f})',
                  fontsize=12, fontweight='bold')
    ax2.set_ylim([0, 1])
    ax2.grid(axis='y', alpha=0.3)

    ax3 = axes[0, 2]
    n_classes = targets.shape[1]
    per_class_f1 = []
    for c in range(n_classes):
        pred_binary = (predictions[:, c] > 0.5).astype(int)
        target_binary = targets[:, c].astype(int)
        f1 = f1_score(target_binary, pred_binary, zero_division=0)
        per_class_f1.append(f1)

    classes = list(range(n_classes))
    bars = ax3.bar(classes, per_class_f1, color='skyblue', alpha=0.7, edgecolor='black')
    ax3.set_xlabel('Class Index', fontsize=11, fontweight='bold')
    ax3.set_ylabel('F1 Score', fontsize=11, fontweight='bold')
    ax3.set_title(f'Per-class F1 Score (Mean: {np.mean(per_class_f1):.4f})', fontsize=12, fontweight='bold')
    ax3.set_ylim([0, 1])
    ax3.axhline(y=np.mean(per_class_f1), color='red', linestyle='--', label=f'Mean: {np.mean(per_class_f1):.4f}')
    ax3.legend()
    ax3.grid(axis='y', alpha=0.3)

    ax4 = axes[1, 0]
    per_class_ap = []
    for c in range(n_classes):
        try:
            ap = average_precision_score(targets[:, c], predictions[:, c])
            per_class_ap.append(ap)
        except:
            per_class_ap.append(0.0)

    bars = ax4.bar(classes, per_class_ap, color='lightcoral', alpha=0.7, edgecolor='black')
    ax4.set_xlabel('Class Index', fontsize=11, fontweight='bold')
    ax4.set_ylabel('Average Precision', fontsize=11, fontweight='bold')
    ax4.set_title(f'Per-class AP (Mean: {np.mean(per_class_ap):.4f})', fontsize=12, fontweight='bold')
    ax4.set_ylim([0, 1])
    ax4.axhline(y=np.mean(per_class_ap), color='red', linestyle='--', label=f'Mean: {np.mean(per_class_ap):.4f}')
    ax4.legend()
    ax4.grid(axis='y', alpha=0.3)

    ax5 = axes[1, 1]
    ax5.hist(predictions.flatten(), bins=50, color='green', alpha=0.7, edgecolor='black')
    ax5.set_xlabel('Prediction Probability', fontsize=11, fontweight='bold')
    ax5.set_ylabel('Frequency', fontsize=11, fontweight='bold')
    ax5.set_title('Prediction Probability Distribution', fontsize=12, fontweight='bold')
    ax5.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Threshold = 0.5')
    ax5.legend()
    ax5.grid(alpha=0.3)

    ax6 = axes[1, 2]
    pred_binary = (predictions > 0.5).astype(int)
    target_binary = targets.astype(int)

    total_tp = ((pred_binary == 1) & (target_binary == 1)).sum()
    total_fp = ((pred_binary == 1) & (target_binary == 0)).sum()
    total_tn = ((pred_binary == 0) & (target_binary == 0)).sum()
    total_fn = ((pred_binary == 0) & (target_binary == 1)).sum()

    cm = np.array([[total_tn, total_fp], [total_fn, total_tp]])

    sns.heatmap(cm, annot=True, fmt='.0f', cmap='Blues', ax=ax6,
                xticklabels=['Pred Neg', 'Pred Pos'],
                yticklabels=['True Neg', 'True Pos'],
                cbar_kws={'label': 'Count'})
    ax6.set_title('Confusion Matrix (Micro Average)', fontsize=12, fontweight='bold')

    plt.tight_layout()

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, f'{dataset_name}_evaluation_visualization.png')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f" Visualization saved to: {save_path}")

    plt.show()

    print(f"\n{'='*70}")
    print(f" {dataset_name} - Detailed Statistics")
    print(f"{'='*70}")
    print(f"Total samples: {len(predictions)}")
    print(f"Number of classes: {n_classes}")
    print(f"\nOverall Metrics:")
    print(f"  Loss:                     {results['avg_loss']:.4f}")
    print(f"  Accuracy:                 {results['avg_acc']:.4f}")
    print(f"  Per-class Accuracy:       {np.mean(per_class_acc):.4f} +/- {np.std(per_class_acc):.4f}")
    print(f"\n Four Core Metrics:")
    print(f"  Macro AP (mAP):           {results['map_macro']:.4f}")
    print(f"  Micro AP:                     {results['map_micro']:.4f}")
    print(f"  Macro F1 (mF1):           {results['f1_macro']:.4f}")
    print(f"  Micro F1:                     {results['f1_micro']:.4f}")
    print(f"\nPer-class F1 Score:")
    print(f"  Mean:                     {np.mean(per_class_f1):.4f}")
    print(f"  Std:                      {np.std(per_class_f1):.4f}")
    print(f"  Min:                      {np.min(per_class_f1):.4f}")
    print(f"  Max:                      {np.max(per_class_f1):.4f}")
    print(f"\nPer-class AP:")
    print(f"  Mean:                     {np.mean(per_class_ap):.4f}")
    print(f"  Std:                      {np.std(per_class_ap):.4f}")
    print(f"  Min:                      {np.min(per_class_ap):.4f}")
    print(f"  Max:                      {np.max(per_class_ap):.4f}")
    print(f"\nPrediction Statistics:")
    print(f"  Total positive samples:    {target_binary.sum()}")
    print(f"  Predicted as positive:     {pred_binary.sum()}")
    print(f"  True Positives:           {total_tp}")
    print(f"  False Positives:          {total_fp}")
    print(f"  True Negatives:           {total_tn}")
    print(f"  False Negatives:          {total_fn}")
    print(f"{'='*70}\n")


if 'trained_model' in locals():
    val_results = evaluate_full(
        model=trained_model,
        dataset=val_dataset,
        n_way=12,        k_shot=5,
        n_query=4,
        n_episodes=200,
        device=DEVICE
    )

    plot_evaluation_metrics(val_results, dataset_name="Validation_Set", save_dir=LOG_DIR)
else:
    print("\n  Model not trained yet. Please train the model first.")

## 3.16 Save Evaluation Results


In [ ]:
def get_val_category_mapping():
    """Return the category mappings for the validation classes."""
    VAL_LABELS_COCO = [
        'cow', 'dining table', 'zebra', 'sandwich', 'bear', 'toaster',
        'person', 'laptop', 'bed', 'teddy bear', 'baseball bat', 'skis'
    ]
    
    print(f"\n{'='*80}")
    print(f"Validation class to COCO category ID mapping")
    print(f"{'='*80}")
    print(f"\nNumber of validation classes: {len(VAL_LABELS_COCO)}")
    print(f"\n{'Class Index':<15} {'Category Name':<25} {'COCO Category ID':<20}")
    print("-" * 80)
    
    mapping_dict = {}
    for idx, cat_name in enumerate(VAL_LABELS_COCO):
        if cat_name in cat_name_to_id:
            cat_id = cat_name_to_id[cat_name]
            mapping_dict[idx] = {
                'class_index': idx,
                'category_name': cat_name,
                'coco_category_id': cat_id
            }
            print(f"{idx:<15} {cat_name:<25} {cat_id:<20}")
        else:
            print(f"{idx:<15} {cat_name:<25} {'NOT FOUND':<20}")
    
    print(f"\n{'='*80}")
    print(f"Key:")
    print(f"  - Class Index: index used during training/evaluation (0-11)")
    print(f"  - Category Name: official COCO class name")
    print(f"  - COCO Category ID: numeric ID in the COCO dataset")
    print(f"{'='*80}\n")
    
    return mapping_dict


val_category_mapping = get_val_category_mapping()

val_mapping_file = os.path.join(LOG_DIR, "validation_class_mapping.json")
val_mapping_data = {
    'description': 'Validation set class index to COCO category mapping',
    'total_classes': 12,
    'mapping': {str(k): v for k, v in val_category_mapping.items()},
    'metadata': {
        'dataset': 'COCO',
        'split': 'validation',
        'source': 'step1_data_preparation'
    }
}

with open(val_mapping_file, 'w') as f:
    json.dump(val_mapping_data, f, indent=2)

print(f" Mapping saved to: {val_mapping_file}")
val_class_index_to_name = {v['class_index']: v['category_name'] for k, v in val_category_mapping.items()}
val_class_index_to_id = {v['class_index']: v['coco_category_id'] for k, v in val_category_mapping.items()}
val_id_to_class_index = {v['coco_category_id']: v['class_index'] for k, v in val_category_mapping.items()}

print(f"\nMapping dictionary created:")
print(f"  - val_class_index_to_name: Class Index to Category Name")
print(f"  - val_class_index_to_id: Class Index to COCO Category ID")
print(f"  - val_id_to_class_index: COCO Category ID to Class Index")

print(f"\nQuick lookup example:")
print(f"  Class 0: {val_class_index_to_name[0]} (COCO ID: {val_class_index_to_id[0]})")
print(f"  Class 5: {val_class_index_to_name[5]} (COCO ID: {val_class_index_to_id[5]})")
print(f"  Class 11: {val_class_index_to_name[11]} (COCO ID: {val_class_index_to_id[11]})")
print()

In [ ]:
if 'val_results' in locals():
    results_file = os.path.join(LOG_DIR, "lc_protonets_val_results.json")

    results_to_save = {
        'avg_loss': float(val_results['avg_loss']),
        'avg_acc': float(val_results['avg_acc']),
        'per_class_acc': [float(x) for x in val_results['per_class_acc']],
        'map_macro': float(val_results['map_macro']),
        'map_micro': float(val_results['map_micro']),
        'f1_macro': float(val_results['f1_macro']),
        'f1_micro': float(val_results['f1_micro']),
        'n_way': 12,
        'k_shot': 5,
        'n_query': 4,
        'n_episodes': 200,
        'dataset_type': 'validation'
    }

    with open(results_file, 'w') as f:
        json.dump(results_to_save, f, indent=2)

    print(f" Validation results saved to: {results_file}")
    print(f"\nFour core metrics saved:")
    print(f"  mAP (macro AP):  {val_results['map_macro']:.4f}")
    print(f"  Micro AP:       {val_results['map_micro']:.4f}")
    print(f"  mF1 (macro F1):  {val_results['f1_macro']:.4f}")
    print(f"  Micro F1:       {val_results['f1_micro']:.4f}")
else:
    print("\n  No results to save.")

## 3.17 Evaluate on the Novel Test Classes

The 16 novel test classes are not used during training. Evaluation on these classes measures few-shot generalization.


In [ ]:
if 'trained_model' in locals():
    print(f"\n{'='*70}")
    print("TESTING ON NOVEL CLASSES")
    print(f"{'='*70}")
    print(f"The test set contains {NOVEL_N_WAY_FULL} classes not seen during training")
    print("This evaluates few-shot generalization")
    test_results = evaluate_full(
        model=trained_model,
        dataset=novel_dataset,
        n_way=NOVEL_N_WAY_FULL,
        k_shot=5,
        n_query=4,
        n_episodes=200,
        device=DEVICE
    )
    plot_evaluation_metrics(test_results, dataset_name="Test_Set_Novel_Classes", save_dir=LOG_DIR)
    print(f"\nTest Results Summary")
    print(f"Test Loss: {test_results['avg_loss']:.4f}")
    print(f"Test Accuracy: {test_results['avg_acc']:.4f}")
    print(f"Per-class Accuracy: {np.mean(test_results['per_class_acc']):.4f} ? {np.std(test_results['per_class_acc']):.4f}")
    print(f"\nFour Core Metrics:")
    print(f"  Macro AP (mAP):       {test_results['map_macro']:.4f}")
    print(f"  Micro AP:             {test_results['map_micro']:.4f}")
    print(f"  Macro F1 (mF1):       {test_results['f1_macro']:.4f}")
    print(f"  Micro F1:             {test_results['f1_micro']:.4f}")
    print(f"{'='*70}")
    print("\nDetailed test-set diagnostics")
    quick_diagnose(test_results['all_predictions'], test_results['all_targets'])
else:
    print("Model not trained yet. Please train the model first.")


In [ ]:
if 'test_results' in locals():
    test_results_file = os.path.join(LOG_DIR, "lc_protonets_test_results.json")

    test_results_to_save = {
        'avg_loss': float(test_results['avg_loss']),
        'avg_acc': float(test_results['avg_acc']),
        'per_class_acc': [float(x) for x in test_results['per_class_acc']],
        'map_macro': float(test_results['map_macro']),
        'map_micro': float(test_results['map_micro']),
        'f1_macro': float(test_results['f1_macro']),
        'f1_micro': float(test_results['f1_micro']),
        'n_way': NOVEL_N_WAY_FULL,
        'k_shot': 5,
        'n_query': 4,
        'n_episodes': 200,
        'test_type': 'novel_classes',
        'num_novel_classes': NOVEL_N_WAY_FULL,
        'dataset_type': 'test'
    }

    with open(test_results_file, 'w') as f:
        json.dump(test_results_to_save, f, indent=2)

    print(f" Test results saved to: {test_results_file}")
    print(f"\nFour core metrics saved:")
    print(f"  mAP (macro AP):  {test_results['map_macro']:.4f}")
    print(f"  Micro AP:       {test_results['map_micro']:.4f}")
    print(f"  mF1 (macro F1):  {test_results['f1_macro']:.4f}")
    print(f"  Micro F1:       {test_results['f1_micro']:.4f}")
else:
    print("\n  No test results to save.")